# Running this experiment

See README.md for setup, required inputs, and limitations. Run cells in order from a fresh kernel. Saved outputs have been cleared; historical metrics are not newly verified results. Environment installation and external training commands are manual setup steps.

In [ ]:
from project_utils import project_path

# Override these paths for your local datasets. PROJECT_DATA_DIR changes their base.
INPUT_PATHS = {
    "GEOPOTENTIAL_500_NC": project_path("geopotential_500/*.nc"),
    "TEMPERATURE_2M_NC": project_path("temperature_2m/*.nc"),
    "TOA_INCIDENT_SOLAR_RADIATION_NC": project_path("toa_incident_solar_radiation/*.nc"),
}

## Get Packages

In [ ]:
pass  # Environment setup moved to README; use local inputs.

In [ ]:
pass  # Environment setup moved to README; use local inputs.
pass  # Environment setup moved to README; use local inputs.

In [ ]:
import numpy as np

import matplotlib.pyplot as plt


from tensorflow import keras

import xarray as xr

## Get the Data

In [ ]:
z500_dataset_path = INPUT_PATHS["GEOPOTENTIAL_500_NC"]

temp_2m_dataset_path = INPUT_PATHS["TEMPERATURE_2M_NC"]

solar_dataset_path = INPUT_PATHS["TOA_INCIDENT_SOLAR_RADIATION_NC"]

## Create Model

In [ ]:
from experiment_lstm_notebook_geopotential_data import create_encoder

In [ ]:
from experiment_lstm_notebook_geopotential_data import create_decoder

In [ ]:
from experiment_lstm_notebook_geopotential_data import VAE

In [ ]:
latent_dim = 2
batch_size = 32
input_shape = (32, 64, 1)
number_of_filters = (32, 64)
k_shape = (3, 3)
stride = (2, 2)

EPOCHS = 500

In [ ]:
encoder = create_encoder(
    latent_dim,
    batch_size=batch_size,
    input_shape=input_shape,
    number_of_filters=number_of_filters,
    k_shape=k_shape,
    stride=stride,
    LSTM_units=4 * latent_dim,
)
encoder.summary()

In [ ]:
decoder = create_decoder(
    latent_dim,
    batch_size=batch_size,
    input_shape=input_shape,
    number_of_filters=number_of_filters,
    k_shape=k_shape,
    stride=stride,
)
decoder.summary()

## Train Model

In [ ]:
z500 = xr.open_mfdataset(z500_dataset_path, combine="by_coords")

In [ ]:
ds_train = z500.sel(time=slice("2015", "2016"))
ds_test = z500.sel(time=slice("2017", "2018"))

In [ ]:
dataset = np.array(ds_train.z)

In [ ]:
num_samples = (dataset.shape[0] // batch_size) * batch_size + 1

In [ ]:
from experiment_lstm_notebook_geopotential_data import normalize_data

from experiment_lstm_notebook_geopotential_data import expand_data

from experiment_lstm_notebook_geopotential_data import create_shifted_frames

In [ ]:
x, y = create_shifted_frames(expand_data(normalize_data(dataset[:num_samples, :, :])))

In [ ]:
print(x.shape, y.shape)

In [ ]:
vae = VAE(encoder, decoder)
vae.compile(optimizer=keras.optimizers.Adam(1e-3))

In [ ]:
batch_size = 32

In [ ]:
vae.fit(x=x, y=x, epochs=EPOCHS * 2, batch_size=batch_size)

### Training Loss

In [ ]:
plt.plot(vae.history.history["loss"])